# KCL: schemas and checks

KCL is a typed configuration language with schemas, defaults and `check` blocks, and it renders YAML directly. `kcl vet` validates plain data files against a schema, `kcl test` runs `test_*` functions.


In [ ]:
export HOME=/tmp
mkdir -p /source/work/other-lab/kcl && cd /source/work/other-lab/kcl
cat > schema.k <<'KCL'
schema Context:
    env: "dev" | "prod"
    replicas: int = 1
    image: str = "traefik/whoami:v1.11.0"
    check:
        0 < replicas <= 20, "replicas must be between 1 and 20"
        not image.endswith(":latest"), "no floating tags"
KCL
cat > main.k <<'KCL'
import schema

prod = schema.Context {env = "prod", replicas = 3}

deployment = {
    apiVersion = "apps/v1"
    kind = "Deployment"
    metadata.name = "web"
    spec = {
        replicas = prod.replicas
        template.spec.containers = [{name = "web", image = prod.image}]
    }
}
KCL
kcl run main.k


In [ ]:
cd /source/work/other-lab/kcl
sed -i 's/replicas = 3/replicas = 30/' main.k && (kcl run main.k 2>&1 | grep -m2 -i 'replicas\|error') || true; sed -i 's/replicas = 30/replicas = 3/' main.k


In [ ]:
cd /source/work/other-lab/kcl
cat > data.yaml <<'YAML'
env: prod
replicas: 3
image: traefik/whoami:v1.11.0
YAML
kcl vet data.yaml schema.k --format yaml -d Context && echo "data.yaml conforms to Context"
kcl fmt main.k >/dev/null && kcl lint main.k && echo "fmt + lint ok"


In [ ]:
cd /source/work/other-lab/kcl
cat > main_test.k <<'KCL'
import schema

test_prod_replicas = lambda {
    ctx = schema.Context {env = "prod", replicas = 3}
    assert ctx.replicas == 3
}
KCL
kcl test ./... 2>&1 | tail -3


## The same language as a renderer

`examples/07-kcl` in the companion repository is KCL as the seventh renderer of the talk's workload: a `Context` schema with defaults, derived fields and cross-field `check:` rules, the Kubernetes objects typed by the `k8s` module from the KCL registry, and the podinfo chart with its Helm values computed in KCL. Its tests assert that bad contexts are rejected with the rule's message.


In [ ]:
export HOME=/tmp
mkdir -p /source/work && cd /source/work && if [ -d gitops-renderers ]; then git -C gitops-renderers pull -q; else git clone -q --recurse-submodules https://github.com/cznewt/gitops-renderers.git; fi
cd gitops-renderers/examples/07-kcl && sed -n 1,30p contexts/schema.k


In [ ]:
cd /source/work/gitops-renderers/examples/07-kcl
export HOME=/tmp KCL_PKG_PATH=/tmp/kcl-pkg
kcl test ./... 2>&1 | tail -13


In [ ]:
cd /source/work/gitops-renderers
export HOME=/tmp TOOLS_LOCAL=1
just render-kcl >/dev/null 2>&1 && git status --short rendered/kcl && echo "(no diff: the committed output matches)" && python3 scripts/compare.py prod | grep -A2 -- '--- kcl'


Try it: add a check rule that forbids the default namespace, then run kcl test.
